# Week 1 — EDA, Preprocessing & Feature Extraction

**Internship Project: Image Captioning AI**

### Learning Objectives
1. Explore and understand the Flickr8k dataset
2. Perform text preprocessing and vocabulary analysis
3. Visualize image features using ResNet50
4. Extract and cache CNN features for faster training

In [ ]:
# ─── Imports ─────────────────────────────────────────────────────────────────
import os, sys
sys.path.insert(0, os.path.dirname(os.getcwd()))   # allow import from project root

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud
from PIL import Image
import torch
import torchvision.models as models
import torchvision.transforms as T

import config
from src.vocabulary import Vocabulary, tokenise
from src.dataset import load_captions_df

print('PyTorch:', torch.__version__)
print('Device :', config.DEVICE)

## 1. Load & Inspect Dataset

In [ ]:
df = load_captions_df(config.CAPTIONS_FILE)
print(f'Total rows   : {len(df)}')
print(f'Unique images: {df["image"].nunique()}')
print(f'Avg captions per image: {len(df)/df["image"].nunique():.1f}')
df.head(10)

In [ ]:
# Show 8 random images with captions
sample = df.sample(8, random_state=42)
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, (_, row) in zip(axes.flat, sample.iterrows()):
    img_path = os.path.join(config.IMAGES_DIR, row['image'])
    img = Image.open(img_path).convert('RGB')
    ax.imshow(img)
    caption_wrapped = '\n'.join([row['caption'][i:i+30] for i in range(0,len(row['caption']),30)])
    ax.set_title(caption_wrapped, fontsize=7, wrap=True)
    ax.axis('off')
plt.suptitle('Sample Images from Flickr8k', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUTS_DIR, 'sample_images.png'), dpi=120)
plt.show()

## 2. Caption Text Analysis

In [ ]:
# Caption lengths
df['length'] = df['caption'].apply(lambda c: len(tokenise(c)))
print(df['length'].describe())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df['length'], bins=30, color='#7c6aff', edgecolor='white', linewidth=0.5)
axes[0].set_title('Caption Length Distribution')
axes[0].set_xlabel('Number of tokens')
axes[0].set_ylabel('Frequency')

# Word frequency
all_words = [w for cap in df['caption'] for w in tokenise(cap)]
word_freq = Counter(all_words)
top20 = word_freq.most_common(20)
words, freqs = zip(*top20)
axes[1].barh(words[::-1], freqs[::-1], color='#ff6b9d')
axes[1].set_title('Top-20 Most Frequent Words')
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUTS_DIR, 'eda_text.png'), dpi=120)
plt.show()

print(f'\nTotal unique words : {len(word_freq)}')
print(f'Total tokens       : {sum(word_freq.values())}')

In [ ]:
# Word cloud
wc = WordCloud(width=800, height=400, background_color='#0a0a0f',
               colormap='cool', max_words=200)
wc.generate(' '.join(all_words))
plt.figure(figsize=(12, 5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Caption Word Cloud', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUTS_DIR, 'wordcloud.png'), dpi=120)
plt.show()

## 3. Build Vocabulary

In [ ]:
vocab = Vocabulary()
vocab.build(df['caption'].tolist())

# Coverage analysis
cum_freq = np.cumsum([f for _, f in vocab.freq.most_common()])
total = cum_freq[-1]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(cum_freq)+1), cum_freq/total*100, color='#7c6aff', lw=2)
ax.axhline(95, color='#ff6b9d', ls='--', label='95% coverage')
ax.axhline(99, color='#4ade80', ls='--', label='99% coverage')
ax.set_xlabel('Vocabulary Size')
ax.set_ylabel('% of total tokens covered')
ax.set_title('Vocabulary Coverage Curve')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUTS_DIR, 'vocab_coverage.png'), dpi=120)
plt.show()

# Save vocab
vocab.save(os.path.join(config.MODELS_DIR, 'vocab.pkl'))

# Test encode/decode
test_cap = 'a dog runs through the grass'
encoded  = vocab.encode(test_cap)
decoded  = vocab.decode(encoded)
print(f'Original : {test_cap}')
print(f'Encoded  : {encoded}')
print(f'Decoded  : {decoded}')

## 4. CNN Feature Extraction (ResNet50)

In [ ]:
from src.feature_extractor import CNNEncoder

encoder = CNNEncoder(model_name='resnet50', feature_dim=config.EMBED_DIM)
encoder.eval().to(config.DEVICE)

preprocess = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Extract features from 4 sample images and visualise
samples = df.drop_duplicates('image').sample(4, random_state=7)
fig, axes = plt.subplots(2, 4, figsize=(14, 6))

for col, (_, row) in enumerate(samples.iterrows()):
    img = Image.open(os.path.join(config.IMAGES_DIR, row['image'])).convert('RGB')
    axes[0, col].imshow(img)
    axes[0, col].set_title(row['caption'][:40], fontsize=7)
    axes[0, col].axis('off')

    tensor = preprocess(img).unsqueeze(0).to(config.DEVICE)
    with torch.no_grad():
        feat = encoder(tensor).cpu().numpy().squeeze()
    axes[1, col].bar(range(50), feat[:50], color='#7c6aff')
    axes[1, col].set_title(f'Feature vector (first 50 dims)', fontsize=7)
    axes[1, col].tick_params(labelsize=6)

plt.suptitle('Image → CNN Feature Vectors', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUTS_DIR, 'feature_extraction.png'), dpi=120)
plt.show()

print(f'\nFeature vector shape: {feat.shape}')
print('Week 1 complete ✓')